In [ ]:
%matplotlib inline
import os, sys
base_dir = "../../"
sys.path.extend([f"{base_dir}/models", 
                 f"{base_dir}/utils", 
                 f"{base_dir}/handle_data", 
                 f"{base_dir}/postprocess"])
import numpy as np
import xarray as xr
import pandas as pd
from scores_class import Scores
from evaluation_utils import perform_block_bootstrap_metric as bootstrapping
# for plotting
import matplotlib as mpl
import matplotlib.pyplot as plt

In [ ]:
cf  = {}
cf['with_snow'] = False
cf['results_basedir'] = "/p/scratch/deepacf/maelstrom/maelstrom_data/ap5/downscaling_benchmark_dataset/benchmark_t2m/results"
cf['data_dir'] = '/p/scratch/deepacf/maelstrom/maelstrom_data/ap5/downscaling_benchmark_dataset/benchmark_t2m/dataset/{}'.format(
                        'with_snow' if cf['with_snow'] else 'without_snow')
cf['experiments'] = ["sha_unet_benchmark_t2m", "sha_wgan_benchmark_t2m", "ankit_swinir", "bilinear"] #"deepru_benchmark_t2m" output file is missing
cf['experiment_name'] = "t2m"
cf['ref_experiment'] = "bilinear"
cf['nboots'] = 1000
cf['metrics'] = ["rmse"]

cf['plot_fname'] = "{}_meta_postprocessing.png"


In [ ]:
class Config:
    def __init__(self,config_dict : dict):
        self.__dict__.update(config_dict)
config = Config(cf)

In [ ]:
rmse_tall_exps= {}
skill_tall_exps = {}

In [ ]:
def filename(exp_name : str):
    if exp_name == "bilinear":
        filename = "downscaling_benchmark_t2m_test.nc"
    else:
        filename = "postprocessed_ds_test.nc"
        if not os.path.exists(os.path.join(config.results_basedir,exp_name,filename)):
            experiment_type = exp_name.replace(f"_benchmark_{config.experiment_name}","")
            filename = f"downscaled_{config.experiment_name}_{experiment_type}.nc"
    return filename
    

def input_target_key_determiner(keys):
    forecast_key = list(filter(lambda key: "in" in key or "fcst" in key,keys))[0]
    target_key = list(filter(lambda key: "tar" in key or "ref" in key,keys))[0]
    return forecast_key,target_key

for i, exp in enumerate(config.experiments):
    print(f"Run evaluation for {exp}...")
    ds = xr.open_dataset(
        os.path.join(
            config.data_dir if exp=="bilinear" else os.path.join(config.results_basedir,exp), 
            filename(exp)))
    forecast_key,target_key = input_target_key_determiner(ds.keys())
    score_engine = Scores(ds[forecast_key], ds[target_key], ["rlat", "rlon"])
                    
    rmse_tall_exps[exp] = score_engine(config.metrics[0])

In [ ]:
# get reference
rmse_ref = rmse_tall_exps[config.ref_experiment]

# initialize DataArrays to store results

config.experiments.remove(config.ref_experiment)
nexps = len(config.experiments)

skill_avg = xr.DataArray(np.zeros(nexps), coords={"experiment": config.experiments}, dims="experiment")
skill_boot = xr.DataArray(np.zeros(nexps*config.nboots).reshape(nexps, config.nboots),
                          coords={"experiment": config.experiments, "iboot": np.arange(config.nboots)},
                          dims=["experiment", "iboot"])

for i, exp in enumerate(config.experiments):
    skill_tall_exps[exp] = (rmse_ref - rmse_tall_exps[exp])/rmse_ref
    skill_avg[i], skill_boot[i,:] = skill_tall_exps[exp].mean(), bootstrapping(skill_tall_exps[exp], "time", 120)

In [ ]:
def plot_skills(skill_avg, skill_boot, plt_fname, labels=["U-Net (Sha)", "WGAN (Sha)", "DeepRU", "SwinIR"],
                metric="RMSE"):
    fs = 16

    # create figure
    fig, ax = plt.subplots(1, 1)
    # create box-plot
    bp = ax.boxplot(skill_boot.T, labels=labels, patch_artist=True)
    # configure plot
    #ax.set_ylim(0, 1.0)

    ax.set_title("")
    ax.set_ylabel(f"Skill {metric}", fontsize=fs)
    ax.tick_params(axis="both", which="both", direction="out", labelsize=fs-2)

    colors = ['pink', 'lightblue', 'lightgreen']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)

    for median in bp['medians']:
        median.set_color('black')
        median.set_linewidth(2.)
        
    plt.rcParams['text.usetex'] = True
    ax.text(0.7, 0.05, r"$\overline{RMSE}_{ref}$="+f"{rmse_ref.mean():.2f}K",
        transform=ax.transAxes,
        color='k', fontsize=fs-2)

    fig.savefig(plt_fname, bbox_inches="tight")
    plt.tight_layout()
    fig.savefig(plt_fname)
    plt.close(fig)